# 🗻 Recreating the Crustal Thickness Globe (Airy Isostasy)

This notebook demonstrates how to create a 3D-printable crustal thickness model (Section 3.2.1 of Koelemeijer & Winterbourne 2021). The model represents how the thickness of the Earth's crust varies between oceans and mountain ranges.

### 🌎 Scientific Context
According to the **Airy Isostasy** model (which explains buoyancy in the Earth's mantle):
- High mountains are supported by deep, low-density crustal "roots" extending down into the mantle (much like a large iceberg has a deep root underwater).
- Ocean basins have thin crust because they sit low.

In this notebook, we load Earth's surface topography and use the Airy Isostasy equation to calculate Moho depth (the boundary between the crust and mantle) and model it as a physical split globe you can open to inspect.

## Step 1: Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    calculate_displacement_scale
)

## Step 2: Load Surface Topography and Calculate Airy Moho Depth

We calculate the Moho depth $d_{Moho}$ directly from the surface topography $h$ using the standard isostasy density ratios:
$$d_{Moho} = d_{reference} + h \times \frac{\rho_c}{\rho_m - ho_c}$$
where $d_{reference} = 30\text{ km}$ (standard average continental crust), density of crust $\rho_c = 2.7\text{ g/cm}^3$, and density of mantle $\rho_m = 3.3\text{ g/cm}^3$.

In [ ]:
# 1. Load ETOPO surface topography
full_grid = GeographicGrid.from_netcdf(
    "../../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z'
)
grid_ds = GeographicGrid(
    lats=full_grid.lats[::15],
    lons=full_grid.lons[::15],
    grid=full_grid.grid[::15, ::15]
)

# 2. Calculate simulated Moho depth using Airy Isostasy
rho_c = 2.7
rho_m = 3.3
ref_thickness = 30000.0  # 30 km in meters
isostasy_ratio = rho_c / (rho_m - rho_c)  # 4.5

# Moho depth is negative (downward from sea level)
moho_depth_grid = - (ref_thickness + grid_ds.grid * isostasy_ratio)
moho_grid = GeographicGrid(lats=grid_ds.lats, lons=grid_ds.lons, grid=moho_depth_grid)
print(f"Calculated Moho grid. Maximum crustal root depth: {np.min(moho_depth_grid) / 1000.0:.1f} km")

## Step 3: Build the Double-Shell Globe Model

We create a hollow sphere where the outer shell is displaced by surface topography and the inner shell represents the Moho boundary (scaled with the same 50× exaggeration so they are physically consistent).

In [ ]:
model_radius_mm = 40.0
vertical_exagg = 50.0

# Initialize a hollow model
model = GlobeModel(
    n_points=5000,
    radius=model_radius_mm,
    hollow=True,
    inner_ratio=0.75, # Sets the average boundary location of the Moho shell
)

# Calculate scale factor (ETOPO data is in meters)
scale = calculate_displacement_scale(model_radius_mm, vertical_exagg=vertical_exagg, grid_units='m')

# Displace outer shell with surface topography
model.outer.displace(GridDisplacer(grid_ds), scale=scale)

# Displace inner shell with Moho depth
# Note: since inner shell vertices point inwards, we use the negative Moho values scaled properly
model.inner.displace(GridDisplacer(moho_grid), scale=scale)
print("Displaced both outer (topography) and inner (Moho) shells.")

## Step 4: Export Hemispheres with Magnet Joint Cavities

We configure magnet joint cavities on the mating surfaces and export the model halves. This allows the globe to be snapped together and opened up to feel the crustal roots by hand.

In [ ]:
model.configure_magnets(
    diameter=5.0,
    height=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    n_magnets=3,
    add_bosses=True,
)

os.makedirs('../../outputs', exist_ok=True)
model.export_hemispheres(
    "../../outputs/paper_crust_top.stl",
    "../../outputs/paper_crust_bottom.stl",
    engine='manifold'
)
print("Watertight crustal thickness hemispheres exported!")